[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/37_uncertainty_and_calibration/90_exercises.ipynb)

# Bài tập — Chương 37: Uncertainty & Calibration (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/37_uncertainty_and_calibration

In [ ]:
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_customer_churn

warnings.filterwarnings("ignore")


def ece_uniform(p, y, n_bins=10):
    bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    return sum((bins == b).mean() * abs(y[bins == b].mean() - p[bins == b].mean()) for b in range(n_bins) if (bins == b).any())


def conformal_quantile(scores, alpha):
    """Điểm số nhỏ thứ ⌈(n + 1)(1 − α)⌉ (vô cùng nếu vượt n)."""
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    return np.inf if k > n else np.sort(scores)[k - 1]

## L0 — Nhớ khái niệm

**Câu 1.** (a) Calibration là gì; vì sao AUC không đo được? (b) ECE và phân rã Brier. (c) Platt vs isotonic. (d) Temperature scaling. (e) Aleatoric vs epistemic. (f) Đảm bảo của split conformal và giả định của nó.

✍️ Trả lời:

**Đáp án:**
(a) Trong các trường hợp dự đoán xác suất $p$, tỉ lệ thật bằng $p$. AUC chỉ phụ thuộc thứ tự — biến đổi đơn điệu giữ AUC nhưng làm hỏng xác suất.
(b) ECE: trung bình có trọng số |tỉ lệ thật − dự đoán TB| theo bin. Brier ≈ reliability − resolution + uncertainty.
(c) Platt: sigmoid 2 tham số trên logit điểm số — ổn định khi ít dữ liệu, chỉ sửa sai lệch dạng sigmoid. Isotonic: hàm bậc thang đơn điệu tuỳ ý (PAV) — linh hoạt, cần nhiều dữ liệu.
(d) Chia logit cho $T$ chọn theo log-loss trên tập calibration; $T > 1$ giảm tự tin; argmax không đổi.
(e) Aleatoric: nhiễu vốn có, không giảm khi thêm dữ liệu; epistemic: thiếu hiểu biết của mô hình, giảm khi thêm dữ liệu, lớn ở vùng thiếu dữ liệu/OOD.
(f) $\mathbb P(y \in C(x)) \ge 1 - \alpha$ (biên, trên cả quá trình chọn calibration và điểm mới) cho mọi mô hình và phân phối, chỉ cần dữ liệu calibration và điểm mới hoán đổi được.

## L1 — Giải thích bằng lời

**Câu 2.** Một bệnh viện xây mô hình dự đoán tái nhập viện trong 30 ngày (tỉ lệ thật 8%), AUC = 0.85. Nhóm (1) cân bằng dữ liệu bằng undersampling về 50/50 khi train; (2) calibrate bằng isotonic **trên chính tập train**; (3) báo cho bác sĩ "nguy cơ của bệnh nhân này là 62%"; (4) dùng một khoảng conformal chung cho thời gian nằm viện của mọi khoa; (5) triển khai sang bệnh viện khác. Chỉ ra vấn đề và cách sửa.

✍️ Trả lời:

**Gợi ý:** (1) Undersampling 50/50 thổi phồng xác suất (tỉ lệ nền 8% → mô hình nghĩ 50%) → điều chỉnh theo tỉ lệ nền (bài 02 §7) hoặc calibrate trên dữ liệu có tỉ lệ thật. (2) Calibrate trên tập train: điểm số trên train quá khớp (gần 0/1) → ánh xạ sai trên dữ liệu mới (bài tập 7); phải dùng tập calibration riêng hoặc `CalibratedClassifierCV`. (3) "62%" chỉ có nghĩa nếu calibrated — kiểm tra reliability diagram/ECE trên dữ liệu giữ lại; truyền đạt kèm khoảng bất định và ngữ cảnh (so với tỉ lệ nền 8%). (4) Một $\hat q$ chung: phủ biên 90% nhưng khoa có độ biến động lớn bị phủ thiếu → conformal theo nhóm (Mondrian) hoặc CQR. (5) Bệnh viện khác = phân phối khác (tỉ lệ nền, quần thể, quy trình) → calibration và đảm bảo conformal có thể mất; calibrate lại trên dữ liệu địa phương, giám sát calibration/độ phủ sau triển khai.

## L2 — Tính tay

**Câu 3.** 6 dự đoán và nhãn: $\hat p = [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]$, $y = [0, 0, 1, 1, 1, 0]$. Với 2 bin $[0, 0.5)$ và $[0.5, 1]$, tính `ece` và `brier`.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
ece = 0.5 * abs(1 / 3 - 0.2) + 0.5 * abs(2 / 3 - 0.8)
brier = np.mean((np.array([0.1, 0.2, 0.3, 0.7, 0.8, 0.9]) - np.array([0, 0, 1, 1, 1, 0])) ** 2)
### END CODE HERE ###
print(f"ECE = {ece:.4f} | Brier = {brier:.4f}")

In [ ]:
p3, y3 = np.array([0.1, 0.2, 0.3, 0.7, 0.8, 0.9]), np.array([0, 0, 1, 1, 1, 0])
check_close(ece, ece_uniform(p3, y3, n_bins=2), message="ECE 2 bin")
check_close(brier, (0.01 + 0.04 + 0.49 + 0.09 + 0.04 + 0.81) / 6, message="Brier")

**Câu 4.** (a) Điểm số calibration (đã sắp xếp) $[0.1, 0.2, \dots, 0.9]$ ($n = 9$), $\alpha = 0.2$. Phân vị conformal là điểm số thứ $\lceil (n + 1)(1 - \alpha)\rceil$ nhỏ nhất → `q4`. (b) Mô hình train trên dữ liệu 50/50 cho $p = 0.6$; tỉ lệ nền thật 10%. Xác suất điều chỉnh `p_adj`.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
q4 = 0.8                                                               # ⌈10 · 0.8⌉ = 8 → điểm số thứ 8
p_adj = 0.6 * (0.1 / 0.5) / (0.6 * (0.1 / 0.5) + 0.4 * (0.9 / 0.5))
### END CODE HERE ###
print(q4, round(p_adj, 4))

In [ ]:
check_close(q4, conformal_quantile(np.arange(1, 10) / 10, 0.2), message="phân vị conformal")
check_close(p_adj, 0.12 / (0.12 + 0.72), message="điều chỉnh tỉ lệ nền: 1/7")

## L3 — Code

**Bài 5.** ECE với bin **chia đều** dễ có bin gần như rỗng khi xác suất dồn về một vùng. Cài `ece_adaptive(p, y, n_bins)` dùng bin **cùng số mẫu** (theo phân vị của $\hat p$). Kiểm tra: xác suất calibrated cho ECE nhỏ; xác suất quá tự tin cho ECE lớn.

In [ ]:
def ece_adaptive(p, y, n_bins=10):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return None
    order = np.argsort(p)
    total = 0.0
    for chunk in np.array_split(order, n_bins):
        total += len(chunk) / len(p) * abs(y[chunk].mean() - p[chunk].mean())
    return total
    ### END CODE HERE ###

In [ ]:
r5 = np.random.default_rng(0)
p_true5 = r5.beta(1, 8, 20000)                                         # xác suất dồn về gần 0 (sự kiện hiếm)
y5 = (r5.random(20000) < p_true5).astype(int)
z = np.log(p_true5 / (1 - p_true5))
p_over = 1 / (1 + np.exp(-2.5 * z))                                    # quá tự tin
print(f"calibrated: ECE đều = {ece_uniform(p_true5, y5):.4f} | ECE thích ứng = {ece_adaptive(p_true5, y5):.4f}")
print(f"quá tự tin: ECE đều = {ece_uniform(p_over, y5):.4f} | ECE thích ứng = {ece_adaptive(p_over, y5):.4f}")
check_less(ece_adaptive(p_true5, y5), 0.01, "calibrated → ECE thích ứng nhỏ")
check_greater(ece_adaptive(p_over, y5), 0.03, "quá tự tin → ECE thích ứng lớn")
check_close(ece_adaptive(np.full(100, 0.3), np.r_[np.ones(30), np.zeros(70)].astype(int), n_bins=1), 0.0, tol=1e-9, message="1 bin: dự đoán hằng = tỉ lệ thật → 0")

**Bài 6.** Cài `fit_temperature(logits, y)` tìm $T$ cực tiểu log-loss bằng `scipy.optimize.minimize_scalar` (tối ưu trên $\log T$ để $T > 0$). Áp dụng cho MLP quá tự tin trên chữ số có 20% nhãn nhiễu; so với tìm kiếm lưới.

In [ ]:
def softmax_T(logits, T):
    z = logits / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return 1.0
    nll = lambda log_T: log_loss(y, softmax_T(logits, np.exp(log_T)), labels=range(logits.shape[1]))
    res = minimize_scalar(nll, bounds=(np.log(0.05), np.log(50)), method="bounded")
    return float(np.exp(res.x))
    ### END CODE HERE ###

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16
r6 = np.random.default_rng(0)
y_noisy = yd.copy()
flip = r6.random(len(yd)) < 0.2
y_noisy[flip] = r6.integers(0, 10, flip.sum())
d_tr, d_rest = train_test_split(np.arange(len(Xd)), test_size=0.5, random_state=0, stratify=yd)
d_cal, d_te = train_test_split(d_rest, test_size=0.5, random_state=0, stratify=yd[d_rest])
mlp6 = MLPClassifier((256,), alpha=0, max_iter=3000, random_state=0).fit(Xd[d_tr], y_noisy[d_tr])
L_cal, L_te = (np.log(np.clip(mlp6.predict_proba(Xd[i]), 1e-15, 1)) for i in (d_cal, d_te))
T6 = fit_temperature(L_cal, y_noisy[d_cal])
grid = np.linspace(0.5, 10, 191)
T_grid = grid[np.argmin([log_loss(y_noisy[d_cal], softmax_T(L_cal, T), labels=range(10)) for T in grid])]
nll_before, nll_after = log_loss(y_noisy[d_te], softmax_T(L_te, 1.0), labels=range(10)), log_loss(y_noisy[d_te], softmax_T(L_te, T6), labels=range(10))
print(f"T (tối ưu) = {T6:.3f} | T (lưới) = {T_grid:.2f} | log-loss test: {nll_before:.3f} → {nll_after:.3f}")
check_close(T6, T_grid, tol=0.06, message="khớp tìm kiếm lưới")
check_greater(T6, 1.5, "mô hình quá tự tin → T > 1")
check_less(nll_after, nll_before * 0.7, "log-loss giảm mạnh")

## L4 — Debug

**Bài 7.** Một nhóm calibrate rừng ngẫu nhiên bằng isotonic regression nhưng dùng **điểm số trên tập train**. Kết quả trên tập test tệ hơn cả không calibrate. Sửa `calibrate` để dùng tập calibration riêng.

In [ ]:
raw = make_customer_churn(20000, seed=0)
X7 = pd.DataFrame({"tenure": raw.tenure_months, "monthly": raw.monthly_charges, "support_calls": raw.support_calls,
                   "month_to_month": (raw.contract == "month-to-month").astype(int), "fiber": (raw.internet_service == "fiber").astype(int)}).astype(float)
y7 = raw.churn.values
i_tr, i_rest = train_test_split(np.arange(len(X7)), test_size=0.4, random_state=0)
i_cal, i_te = train_test_split(i_rest, test_size=0.5, random_state=0)
forest = RandomForestClassifier(300, min_samples_leaf=1, random_state=0, n_jobs=-1).fit(X7.iloc[i_tr], y7[i_tr])
s_test = forest.predict_proba(X7.iloc[i_te])[:, 1]


def calibrate_buggy():
    iso = IsotonicRegression(out_of_bounds="clip").fit(forest.predict_proba(X7.iloc[i_tr])[:, 1], y7[i_tr])
    return iso.predict(s_test)


def calibrate():
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: return calibrate_buggy()
    iso = IsotonicRegression(out_of_bounds="clip").fit(forest.predict_proba(X7.iloc[i_cal])[:, 1], y7[i_cal])
    return iso.predict(s_test)
    ### END CODE HERE ###


results7 = {"không calibrate": s_test, "isotonic trên train (lỗi)": calibrate_buggy(), "isotonic trên tập calibration": calibrate()}
for k, p in results7.items():
    print(f"{k:30s}: ECE = {ece_uniform(p, y7[i_te]):.4f} | log-loss = {log_loss(y7[i_te], np.clip(p, 1e-6, 1 - 1e-6)):.4f}")
print(f"điểm số rừng trên train: {np.mean((forest.predict_proba(X7.iloc[i_tr])[:, 1] < 0.1) | (forest.predict_proba(X7.iloc[i_tr])[:, 1] > 0.9)):.0%} gần 0 hoặc 1")

In [ ]:
check_greater(ece_uniform(results7["isotonic trên train (lỗi)"], y7[i_te]), ece_uniform(results7["isotonic trên tập calibration"], y7[i_te]) + 0.02, "bản lỗi tệ hơn rõ")
check_less(ece_uniform(results7["isotonic trên tập calibration"], y7[i_te]), 0.03, "calibrate đúng cách: ECE nhỏ")

**Nhận xét:** Rừng với lá 1 mẫu gần như học thuộc tập train: điểm số trên train dồn về 0/1 và gần như luôn "đúng". Isotonic học ánh xạ "điểm 0.9 → gần chắc chắn rời bỏ" từ dữ liệu đó — nhưng trên dữ liệu mới, điểm 0.9 đúng ít hơn nhiều → xác suất sau calibration **quá tự tin**, còn tệ hơn không làm gì. Calibration phải học trên điểm số của dữ liệu **chưa thấy khi train** (tập calibration riêng, hoặc dự đoán out-of-fold — `CalibratedClassifierCV`). Cùng nguyên lý với mọi bước "học" trong pipeline: không fit trên dữ liệu đã dùng để train mô hình nếu bước đó cần phản ánh hành vi trên dữ liệu mới.

## L5 — Phân tích

**Bài 8.** **Cỡ tập calibration và conformal.** Hồi quy tổng hợp $y = x + \varepsilon$ (mô hình $\hat f(x) = x$ cho sẵn, sai số $\varepsilon\sim t_3$ đuôi dày). Với $n_{cal} \in \{20, 100, 1000\}$, lặp 500 lần: lấy $n_{cal}$ điểm calibration, tính $\hat q$ ($\alpha = 0.1$), đo độ phủ trên 2 000 điểm test mới. Lưu `stats8[n] = (độ phủ TB, độ lệch chuẩn của độ phủ, độ rộng TB)`. Nhận xét.

In [ ]:
stats8 = {}
### START CODE HERE ### (≈ 9 dòng)
# STARTER: # với n: 500 lần lặp; scores = |ε_cal|; q = conformal_quantile(scores, 0.1); độ phủ = mean(|ε_test| ≤ q)
r8 = np.random.default_rng(0)
for n in [20, 100, 1000]:
    covs, widths = [], []
    for _ in range(500):
        q8 = conformal_quantile(np.abs(r8.standard_t(3, n)), 0.1)
        covs.append(np.mean(np.abs(r8.standard_t(3, 2000)) <= q8))
        widths.append(2 * q8)
    stats8[n] = (np.mean(covs), np.std(covs), np.mean(widths))
    print(f"n_cal = {n:4d}: độ phủ TB = {stats8[n][0]:.3f} | độ lệch chuẩn độ phủ = {stats8[n][1]:.3f} | độ rộng TB = {stats8[n][2]:.2f}")
### END CODE HERE ###

In [ ]:
for n in stats8:
    check_greater(stats8[n][0], 0.895, f"n = {n}: độ phủ trung bình ≥ 1 − α (đảm bảo đúng với mọi n)")
check_less(stats8[1000][1], stats8[20][1] / 3, "n lớn → độ phủ từng lần ổn định hơn nhiều")

**Nhận xét:** Đảm bảo đúng với **mọi** $n$ — kể cả 20 điểm, kể cả nhiễu đuôi dày (không cần giả định Gaussian). Nhưng đảm bảo là **trung bình qua các lần chọn tập calibration**: với $n$ nhỏ, độ phủ thực tế của **một** lần triển khai dao động mạnh (có lần chỉ ~75%, có lần ~100%) và khoảng hơi bảo thủ, rộng hơn. Độ phủ có điều kiện theo tập calibration tuân theo phân phối Beta; muốn độ phủ thực tế nằm sát $1 - \alpha$ với xác suất cao, cần vài trăm tới vài nghìn điểm calibration (Angelopoulos & Bates, 2023, §3).

## L6 — Thực tế

**Bài 9.** **Dự đoán có chọn lọc.** Hệ thống đọc chữ số tự động chuyển ca **không chắc** cho người kiểm tra. Dùng logistic regression trên chữ số (nhãn sạch). (1) Vẽ đường **accuracy theo tỉ lệ được tự động xử lý** (sắp ca theo độ tự tin giảm dần). (2) Chọn ngưỡng độ tự tin **trên tập calibration** sao cho accuracy các ca tự động ≥ 99.5%; báo cáo trên tập test: tỉ lệ tự động `auto_rate`, accuracy ca tự động `auto_acc`.

In [ ]:
Xd9, yd9 = load_digits(return_X_y=True)
Xd9 = Xd9 / 16
j_tr, j_rest = train_test_split(np.arange(len(Xd9)), test_size=0.5, random_state=1, stratify=yd9)
j_cal, j_te = train_test_split(j_rest, test_size=0.5, random_state=1, stratify=yd9[j_rest])
clf9 = LogisticRegression(max_iter=3000, C=0.5).fit(Xd9[j_tr], yd9[j_tr])
### START CODE HERE ### (≈ 14 dòng)
# STARTER: correct_te = None       # mảng bool: dự đoán đúng trên j_te
# STARTER: coverage_curve = None   # [(f, accuracy của f·100% ca tự tin nhất trên test)] với f ∈ [0.5, 0.8, 0.9, 1.0]
# STARTER: threshold9 = None       # ngưỡng độ tự tin chọn trên j_cal
# STARTER: auto_rate, auto_acc = None, None
def conf_correct(idx):
    P = clf9.predict_proba(Xd9[idx])
    return P.max(1), P.argmax(1) == yd9[idx]


conf_cal, correct_cal = conf_correct(j_cal)
order = np.argsort(-conf_cal)
running_acc = np.cumsum(correct_cal[order]) / np.arange(1, len(order) + 1)
ok = np.flatnonzero(running_acc >= 0.995)
k = ok.max() + 1                                                       # số ca tự động lớn nhất vẫn đạt 99.5% trên calibration
threshold9 = conf_cal[order][k - 1]
conf_te, correct_te = conf_correct(j_te)
auto = conf_te >= threshold9
auto_rate, auto_acc = auto.mean(), correct_te[auto].mean()
coverage_curve = [(f, correct_te[np.argsort(-conf_te)][: max(1, int(f * len(j_te)))].mean()) for f in [0.5, 0.8, 0.9, 1.0]]
### END CODE HERE ###
print(f"accuracy nếu tự động 100% = {correct_te.mean():.3f}")
print("accuracy theo tỉ lệ tự động:", [(f, round(a, 4)) for f, a in coverage_curve])
print(f"ngưỡng độ tự tin = {threshold9:.3f} | tự động {auto_rate:.1%} ca với accuracy {auto_acc:.2%} | chuyển cho người {1 - auto_rate:.1%}")

In [ ]:
check_greater(auto_acc, correct_te.mean(), "các ca tự động chính xác hơn trung bình")
check_greater(auto_rate, 0.6, "vẫn tự động phần lớn ca")
check_greater(auto_acc, 0.985, "accuracy ca tự động gần mục tiêu trên dữ liệu mới")

**Nhận xét:** Độ tự tin cao tương quan với đúng — loại bỏ một phần nhỏ ca khó nhất làm accuracy phần còn lại tăng mạnh (đường risk–coverage). Ngưỡng phải chọn trên **tập calibration** rồi kiểm chứng trên tập test: trên dữ liệu mới, accuracy ca tự động có thể thấp hơn mục tiêu một chút (ước lượng từ vài trăm ca có sai số) — muốn đảm bảo chặt, dùng các phương pháp có đảm bảo thống kê (conformal risk control, Learn-then-Test). Độ tự tin chỉ hữu ích nếu mô hình calibrated tương đối và dữ liệu không trôi.

## L7 — Nâng cao

**Bài 10.** **Conformal thích ứng khi dữ liệu trôi** (Adaptive Conformal Inference — Gibbs & Candès, 2021). Chuỗi thời gian $y_t = 0.8y_{t-1} + \varepsilon_t$; **độ lệch chuẩn nhiễu tăng gấp 3** từ $t = 1500$. Dự đoán $\hat y_t = 0.8y_{t-1}$. (a) **Tĩnh**: $\hat q$ tính một lần từ 500 sai số đầu. (b) **ACI**: dùng cửa sổ 500 sai số gần nhất và mức $\alpha_t$ cập nhật online: $\alpha_{t+1} = \alpha_t + \gamma(\alpha - \text{err}_t)$ ($\text{err}_t = 1$ nếu lỡ, $\gamma = 0.01$). Lưu `cov10[tên] = (độ phủ trước t = 1500, độ phủ sau t = 1700)`.

In [ ]:
r10 = np.random.default_rng(0)
T10 = 3000
sigma10 = np.where(np.arange(T10) < 1500, 1.0, 3.0)
y10 = np.zeros(T10)
for t in range(1, T10):
    y10[t] = 0.8 * y10[t - 1] + r10.normal(0, sigma10[t])
residuals = np.abs(y10[1:] - 0.8 * y10[:-1])                         # sai số tại t = 1..T−1
cov10 = {}
### START CODE HERE ### (≈ 18 dòng)
# STARTER: # (a) q tĩnh từ residuals[:500]; (b) ACI: với t ≥ 500, q_t = phân vị (1 − α_t) của residuals[t−500:t]; cập nhật α_t
alpha, gamma = 0.1, 0.01
q_static = conformal_quantile(residuals[:500], alpha)
hit_static = residuals <= q_static
alpha_t, hits_aci = alpha, np.zeros(len(residuals), dtype=bool)
for t in range(500, len(residuals)):
    window = residuals[t - 500:t]
    q_t = np.inf if alpha_t <= 0 else (0.0 if alpha_t >= 1 else np.quantile(window, 1 - alpha_t))
    hits_aci[t] = residuals[t] <= q_t
    alpha_t = alpha_t + gamma * (alpha - (0.0 if hits_aci[t] else 1.0))
steps = np.arange(1, T10)
before, after = (steps >= 500) & (steps < 1500), steps >= 1700
cov10["tĩnh"] = (hit_static[before].mean(), hit_static[after].mean())
cov10["ACI"] = (hits_aci[before].mean(), hits_aci[after].mean())
### END CODE HERE ###
for k, (b, a) in cov10.items():
    print(f"{k:5s}: độ phủ trước khi trôi = {b:.3f} | sau khi trôi (t ≥ 1700) = {a:.3f}")

In [ ]:
check_greater(cov10["tĩnh"][0], 0.85, "tĩnh: trước khi trôi vẫn phủ ~90%")
check_less(cov10["tĩnh"][1], 0.6, "tĩnh: sau khi nhiễu tăng, độ phủ sụp đổ")
check_greater(cov10["ACI"][1], 0.85, "ACI: khôi phục độ phủ ~90% sau khi trôi")

**Nhận xét:** $\hat q$ tĩnh đúng cho thời kỳ cũ nhưng sụp đổ khi nhiễu tăng gấp 3 — giả định hoán đổi bị phá. ACI dùng cửa sổ trượt (quên dữ liệu cũ) và tự điều chỉnh mức $\alpha_t$: mỗi lần lỡ, $\alpha_t$ giảm → khoảng rộng ra; mỗi lần trúng, $\alpha_t$ tăng nhẹ → khoảng hẹp lại. Đảm bảo của ACI là **dài hạn**: tần suất lỡ trung bình tiến về $\alpha$ bất kể dữ liệu trôi thế nào (kể cả đối kháng), đổi lại khoảng có thể dao động mạnh ngay sau thời điểm trôi. Đây là cách conformal được dùng cho dự báo chuỗi thời gian và giám sát mô hình sau triển khai.